In [ ]:
import chipwhisperer as cw
import numpy as np
import string
import time


# INITIALISATION


scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial2)
target.baud = 9600

POI_START, POI_END = 1690, 2100

CHARSET_STR = string.ascii_letters + string.digits + "- @"
CHARSET = [ord(c) for c in CHARSET_STR]

# CAPTURE


def collect_mean_trace(known_prefix, candidate_byte, n_repeat=15):
    traces = []
    for i in range(n_repeat):
        cmd = known_prefix + bytes([candidate_byte]) + b"\n"
        time.sleep(0.05)
        scope.arm()
        target.ser.write(cmd)
        if not scope.capture():
            traces.append(scope.get_last_trace())
    if not traces:
        return None
    return np.mean(traces, axis=0)

# RECHERCHE DU CARACTÈRE SUIVANT 


def find_next_byte(known_prefix, known_wrong_char="x", charset=CHARSET,
                    n_repeat=15, poi_start=POI_START, poi_end=POI_END):

    template_wrong = collect_mean_trace(known_prefix, ord(known_wrong_char), n_repeat * 2)
    if template_wrong is None:
        print("Impossible de capturer le template 'faux', arrêt")
        return []

    results = {}
    for guess in charset:
        if chr(guess) == known_wrong_char: #skip le mauvais
            continue
        mt = collect_mean_trace(known_prefix, guess, n_repeat)
        if mt is None:
            continue
        dist = np.linalg.norm(
            mt[poi_start:poi_end] - template_wrong[poi_start:poi_end]
        )
        results[guess] = dist

    ranked = sorted(results.items(), key=lambda x: x[1], reverse=True)
    return ranked



ranking = find_next_byte(known_prefix=b"f7-", known_wrong_char="x", n_repeat=15)

print("Meilleurs candidats pour le caractère suivant 'f7-' :")
for byte_val, score in ranking[:10]:
    print(f"  '{chr(byte_val)}'  ->  distance au template faux = {score:.4f}")

Meilleurs candidats pour le caractère suivant 'f7-' :
  '@'  ->  distance au template faux = 1.5725
  'Z'  ->  distance au template faux = 1.4184
  '0'  ->  distance au template faux = 1.4175
  's'  ->  distance au template faux = 1.3986
  '3'  ->  distance au template faux = 1.3762
  'O'  ->  distance au template faux = 1.3755
  '9'  ->  distance au template faux = 1.3705
  '5'  ->  distance au template faux = 1.3685
  'u'  ->  distance au template faux = 1.3678
  'S'  ->  distance au template faux = 1.3583
